# Models

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

from lightgbm import LGBMRanker

# LambaMART

In [8]:
df = pd.read_csv("../data/processed/train_prepared.csv")

df.shape

(1048575, 65)

# Relevance Targets

In [9]:
df["relevance"] = 0
df.loc[df["click_bool"] == 1, "relevance"] = 1
df.loc[df["booking_bool"] == 1, "relevance"] = 5

In [10]:
df = df.sort_values("srch_id").reset_index(drop=True)

In [11]:
unique_searches = df["srch_id"].unique()

train_searches, val_searches = train_test_split(
    unique_searches,
    test_size=0.2,
    random_state=42
)

train_df = df[df["srch_id"].isin(train_searches)].copy()
val_df = df[df["srch_id"].isin(val_searches)].copy()

In [22]:
drop_cols = [
    "srch_id",
    "booking_bool",
    "click_bool",
    "date_time",
    "relevance",
    "origin_destination_distance"
]

features = [col for col in df.columns if col not in drop_cols]

X_train = train_df[features]
y_train = train_df["relevance"]

X_val = val_df[features]
y_val = val_df["relevance"]

In [23]:
train_group = train_df.groupby("srch_id").size().values
val_group = val_df.groupby("srch_id").size().values

In [24]:
model = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    boosting_type="gbdt",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    group=train_group,
    eval_set=[(X_val, y_val)],
    eval_group=[val_group],
    eval_at=[5],
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.124139 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5084
[LightGBM] [Info] Number of data points in the train set: 839092, number of used features: 61


LGBMRanker(learning_rate=0.05, metric='ndcg', n_estimators=500,
           objective='lambdarank', random_state=42)

In [19]:
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(20)

,feature,importance
12,price_usd,1042
11,prop_log_historical_price,1025
10,prop_location_score2,1016
9,prop_location_score1,992
52,price_relative,964
5,prop_id,801
22,orig_destination_distance,689
54,price_rank,661
16,srch_booking_window,520
14,srch_destination_id,477


In [20]:
val_df["pred_score"] = model.predict(X_val)

In [21]:
sample_srch_id = val_df["srch_id"].iloc[0]

val_df[val_df["srch_id"] == sample_srch_id][
    ["srch_id", "prop_id", "relevance", "booking_bool", "click_bool", "pred_score"]
].sort_values("pred_score", ascending=False).head(10)

,srch_id,prop_id,relevance,booking_bool,click_bool,pred_score
36,4,125069,0,0,0,0.528182
54,4,56063,0,0,0,0.323686
31,4,89119,0,0,0,0.192899
53,4,3625,0,0,0,0.167782
28,4,109185,0,0,0,0.142263
29,4,85567,0,0,0,0.020418
45,4,75491,0,0,0,-0.008186
43,4,83045,0,0,0,-0.019563
52,4,46162,0,0,0,-0.044140
37,4,127808,0,0,0,-0.050466


# NDCG

In [25]:
val_df["pred_score"] = model.predict(X_val)


In [26]:
from sklearn.metrics import ndcg_score

ndcg_scores = []

for srch_id, group in val_df.groupby("srch_id"):

    true_relevance = [group["relevance"].values]
    predicted_scores = [group["pred_score"].values]

    score = ndcg_score(
        true_relevance,
        predicted_scores,
        k=5
    )

    ndcg_scores.append(score)

mean_ndcg = np.mean(ndcg_scores)

print(f"Mean NDCG@5: {mean_ndcg:.4f}")

Mean NDCG@5: 0.3816
